In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, mean_absolute_error, r2_score
)
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred) ** 0.5

from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.datasets import fetch_california_housing
from sklearn.pipeline import make_pipeline


In [ ]:
def charger_immobilier():
  
    data = fetch_california_housing()
    X = data.data
    y = data.target 

    print(f"California Housing : {X.shape[0]} lignes, {X.shape[1]} variables")
    print(f"Variables : {list(data.feature_names)}")
    print(f"Cible : prix médian en centaines de milliers de $")
    print(f"  min={y.min():.2f}, max={y.max():.2f}, moyenne={y.mean():.2f}")

    return X, y


def evaluer_regression(nom_modele, modele, X_train, X_test, y_train, y_test):
  
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)

    print(f"{nom_modele:<22} : R2={r2:.2f}  MAE={mae:.2f}  RMSE={rmse:.2f}")
    return {"r2": r2, "mae": mae, "rmse": rmse}


X_immo, y_immo = charger_immobilier()

X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_immo, y_immo, test_size=0.2, random_state=42
)
scaler_i = StandardScaler()
X_tr_is = scaler_i.fit_transform(X_tr_i)
X_te_is = scaler_i.transform(X_te_i)

print("\n--- Résultats ---")
res_lr = evaluer_regression(
    "LinearRegression",
    LinearRegression(), X_tr_is, X_te_is, y_tr_i, y_te_i
)
res_rf = evaluer_regression(
    "RandomForest",
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_tr_is, X_te_is, y_tr_i, y_te_i
)


In [ ]:


data_immo = fetch_california_housing()
lr_interp = LinearRegression().fit(X_tr_is, y_tr_i)

print("Coefficients de la régression linéaire :")
print("(+coef = augmente le prix | -coef = baisse le prix)")
print()
for nom, poids in zip(data_immo.feature_names, lr_interp.coef_):
    print(f"  {nom:>15} : {poids:+.3f}")


In [ ]:

lr_small = LinearRegression()
lr_small.fit(X_tr_is[:100], y_tr_i[:100])
r2_small = r2_score(y_te_i, lr_small.predict(X_te_is))
print(f"R2 avec 100 lignes seulement : {r2_small:.3f}")
print(f"R2 avec le dataset complet   : {res_lr['r2']:.3f}")



quartier_fictif = np.zeros((1, X_immo.shape[1]))
quartier_fictif[0, 0] = 0.0  
quartier_fictif[0, 4] = 9000 

quartier_fictif_scaled = scaler_i.transform(quartier_fictif)
prix_predit = lr_interp.predict(quartier_fictif_scaled)[0]

print(f"Quartier fictif (revenu=0, pop=9000) → prix prédit : {prix_predit:.2f} (×100k$)")
